# 01 데이터 인덱스 EDA - Phase 1

**목적**: `build_sample_index()` 출력 분포 검증 + `CSVLoader.load()` 단일 호출 스모크 + 결측 폴더 점검.

검증 항목:
1. (vehicle x sensor x fault_class) unique 조합 수 = 45 (V1-1)
2. 임의 CSV 1개 로드 → signal_df shape (10000, 4), meta_df 컬럼 확인 (V1-2)
3. 결측 폴더 감지 시 WARN 로그 (V1-3)

## (Colab 전용) 의존성 설치

In [1]:
# !pip install -q -r ../src/requirements.txt

## 1. import 및 경로 설정

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd

from src.common.seed import set_global_seed
from src.data import CSVLoader, LABEL_MAP, build_sample_index

set_global_seed(42)

DATA_ROOT = Path(PROJECT_ROOT) / "샘플데이터" / "03.합성데이터" / "1.모터_감속기_시계열"
ARTIFACTS_DIR = Path(PROJECT_ROOT) / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
print(f"DATA_ROOT exists: {DATA_ROOT.exists()}")
print(f"LABEL_MAP: {LABEL_MAP}")

DATA_ROOT exists: True
LABEL_MAP: {'NORMAL': 0, 'ECC10': 1, 'ECC20': 2, 'DEMAG': 3, 'REDUC': 4}


## 2. 샘플 인덱스 빌드

In [3]:
idx = build_sample_index(DATA_ROOT)
print(f"총 샘플 수: {len(idx):,}")
print(f"unique group_key: {idx['group_key'].nunique():,}")
idx.head()

2026-05-16 21:32:55,755 [INFO] data.path_indexer: index built: total=81 rows, leaves_scanned=45, leaves_empty=0, unique(vehicle,sensor,fault)=45


총 샘플 수: 81
unique group_key: 27


,csv_path,vehicle,fault_class,date,sensor,group_key
0,C:\Users\kingm\OneDrive\문서\새 폴더\OneDrive\바탕 화면...,IONIQ,NORMAL,0609,Current_U,22_06_09_12_26_53_01_zsplit001
1,C:\Users\kingm\OneDrive\문서\새 폴더\OneDrive\바탕 화면...,IONIQ,NORMAL,0609,Vib_Motor,22_06_09_12_26_53_01_zsplit001
2,C:\Users\kingm\OneDrive\문서\새 폴더\OneDrive\바탕 화면...,IONIQ,NORMAL,0609,Vib_TM,22_06_09_12_26_53_01_zsplit001
3,C:\Users\kingm\OneDrive\문서\새 폴더\OneDrive\바탕 화면...,IONIQ,ECC10,0707,Current_U,22_07_07_18_28_34_01_zsplit001
4,C:\Users\kingm\OneDrive\문서\새 폴더\OneDrive\바탕 화면...,IONIQ,ECC10,0707,Vib_Motor,22_07_07_18_28_34_01_zsplit001


## 3. V1-1 — (vehicle × sensor × fault_class) unique = 45

In [4]:
ngroups = idx.groupby(["vehicle", "sensor", "fault_class"], observed=True).ngroups
print(f"ngroups = {ngroups}")
assert ngroups == 45, f"V1-1 FAIL: ngroups={ngroups}"
print("V1-1 PASS")

ngroups = 45
V1-1 PASS


## 4. 분포 표 — (차종 × 센서 × 고장)별 샘플 수

In [5]:
pv_vehicle_fault = idx.pivot_table(
    index="vehicle", columns="fault_class", values="csv_path",
    aggfunc="count", fill_value=0,
)
print("[vehicle x fault_class]")
pv_vehicle_fault

[vehicle x fault_class]


fault_class,DEMAG,ECC10,ECC20,NORMAL,REDUC
vehicle,,,,,
IONIQ,3,3,6,3,12
KONA,3,6,3,6,6
NIRO,3,9,9,3,6


In [6]:
pv_sensor_fault = idx.pivot_table(
    index="sensor", columns="fault_class", values="csv_path",
    aggfunc="count", fill_value=0,
)
print("[sensor x fault_class]")
pv_sensor_fault

[sensor x fault_class]


fault_class,DEMAG,ECC10,ECC20,NORMAL,REDUC
sensor,,,,,
Current_U,3,6,6,4,8
Vib_Motor,3,6,6,4,8
Vib_TM,3,6,6,4,8


In [7]:
pv_full = idx.pivot_table(
    index=["vehicle", "sensor"], columns="fault_class", values="csv_path",
    aggfunc="count", fill_value=0,
)
print("[vehicle x sensor x fault_class] - 45 행 기대")
pv_full

[vehicle x sensor x fault_class] - 45 행 기대


fault_class        DEMAG  ECC10  ECC20  NORMAL  REDUC
vehicle sensor                                       
IONIQ   Current_U      1      1      2       1      4
        Vib_Motor      1      1      2       1      4
        Vib_TM         1      1      2       1      4
KONA    Current_U      1      2      1       2      2
        Vib_Motor      1      2      1       2      2
        Vib_TM         1      2      1       2      2
NIRO    Current_U      1      3      3       1      2
        Vib_Motor      1      3      3       1      2
        Vib_TM         1      3      3       1      2

## 5. V1-2 — 임의 CSV 1개 로드 (shape & meta 컬럼 확인)

In [8]:
loader = CSVLoader()
sample_path = idx["csv_path"].iloc[0]
print(f"loading: {sample_path}")
meta_df, signal_df = loader.load(sample_path)

print(f"signal_df.shape: {signal_df.shape}")
print(f"signal_df.dtypes: \n{signal_df.dtypes}")
print(f"meta_df:")
meta_df

loading: C:\Users\kingm\OneDrive\문서\새 폴더\OneDrive\바탕 화면\학교\4학년 1학기\시스템분석\샘플데이터\03.합성데이터\1.모터_감속기_시계열\IONIQ\NORMAL\0609\Current_U\22_06_09_12_26_53_01_zsplit001_Current_U_timeseries.csv
signal_df.shape: (10000, 4)
signal_df.dtypes: 
peak_freq_bin    float32
band_start       float32
band_end         float32
rms              float32
dtype: object
meta_df:


,group_key,vehicle,fault_class,date,sensor,timestamp,zsplit,label_id
0,22_06_09_12_26_53_01_zsplit001_Current_U,IONIQ,NORMAL,609,Current_U,22_06_09_12_26_53,zsplit001,0


In [9]:
assert signal_df.shape == (10000, 4), f"V1-2 FAIL: shape={signal_df.shape}"
required_meta = {"vehicle", "sensor", "fault_class", "group_key", "zsplit"}
assert required_meta.issubset(set(meta_df.columns)), \
    f"V1-2 FAIL: missing {required_meta - set(meta_df.columns)}"
assert all(str(signal_df[c].dtype) == "float32" for c in signal_df.columns), \
    "V1-2 FAIL: signal dtype is not float32"
print("V1-2 PASS")

V1-2 PASS


## 6. V1-3 — 결측 폴더 시나리오 WARN 로그 확인

In [10]:
import io
import logging
from src.common.logger import get_logger

logger = get_logger("data.path_indexer")
buf = io.StringIO()
handler = logging.StreamHandler(buf)
handler.setLevel(logging.WARNING)
logger.addHandler(handler)
try:
    _ = build_sample_index(
        DATA_ROOT,
        fault_classes=("NORMAL", "ECC10", "ECC20", "DEMAG", "REDUC", "__GHOST__"),
    )
finally:
    logger.removeHandler(handler)
warn_lines = [ln for ln in buf.getvalue().splitlines() if "missing fault_dir" in ln]
print(f"WARN lines emitted: {len(warn_lines)}")
for ln in warn_lines:
    print("  ", ln)
assert len(warn_lines) >= 1, "V1-3 FAIL: no WARN emitted for missing folder"
print("V1-3 PASS")

2026-05-16 21:32:55,917 [WARNING] data.path_indexer: missing fault_dir: IONIQ/__GHOST__ (skipped)


2026-05-16 21:32:55,932 [WARNING] data.path_indexer: missing fault_dir: KONA/__GHOST__ (skipped)


2026-05-16 21:32:55,950 [WARNING] data.path_indexer: missing fault_dir: NIRO/__GHOST__ (skipped)


2026-05-16 21:32:55,953 [INFO] data.path_indexer: index built: total=81 rows, leaves_scanned=45, leaves_empty=0, unique(vehicle,sensor,fault)=45


WARN lines emitted: 3
   missing fault_dir: IONIQ/__GHOST__ (skipped)
   missing fault_dir: KONA/__GHOST__ (skipped)
   missing fault_dir: NIRO/__GHOST__ (skipped)
V1-3 PASS


## 7. 인덱스 캐시 저장 (`artifacts/sample_index.parquet`)

In [11]:
out_path = ARTIFACTS_DIR / "sample_index.parquet"
idx.to_parquet(out_path, index=False)
print(f"saved: {out_path}  ({out_path.stat().st_size:,} bytes)")

# 라운드트립 검증
loaded = pd.read_parquet(out_path)
assert len(loaded) == len(idx)
assert list(loaded.columns) == list(idx.columns)
print("parquet roundtrip OK")

saved: C:\Users\kingm\OneDrive\문서\새 폴더\OneDrive\바탕 화면\학교\4학년 1학기\시스템분석\artifacts\sample_index.parquet  (8,349 bytes)

parquet roundtrip OK


---
모든 셀이 통과하면 **Phase 1 완료**. Phase 2 (전처리: shape/NaN/panel) 로 진행.